In [1]:
%pip install -q pixeltable-pgserver "psycopg[binary]>=3.1" "sqlalchemy>=2.0" pandas


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

try:
    import pgserver
except ImportError:
    import pixeltable_pgserver as pgserver

pg = pgserver.get_server('/tmp/goit_hw2_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

def execute(*statements):
    """Execute SQL statements in one committed transaction."""
    with engine.begin() as conn:
        for statement in statements:
            conn.execute(text(statement))

def show(query, params=None):
    """Display SELECT results as a DataFrame."""
    with engine.connect() as conn:
        frame = pd.read_sql_query(text(query), conn, params=params)
    display(frame)
    return frame

execute("SET TIME ZONE 'UTC'")
show("SELECT current_setting('TimeZone') AS timezone, version() AS postgresql_version")

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:467: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


,timezone,postgresql_version
0,UTC,"PostgreSQL 16.15 on x86_64-pc-linux-gnu, compi..."


,timezone,postgresql_version
0,UTC,"PostgreSQL 16.15 on x86_64-pc-linux-gnu, compi..."


In [3]:
ddl = [
    """DROP TABLE IF EXISTS
        hw2_user_features_online,
        hw2_training_events,
        hw2_user_features_offline,
        hw2_movie_genres,
        hw2_ratings,
        hw2_genres,
        hw2_movies,
        hw2_users
    CASCADE""",
    """CREATE TABLE hw2_users (
        user_id BIGINT PRIMARY KEY,
        signup_date DATE NOT NULL,
        country VARCHAR(2) NOT NULL CHECK (char_length(country) = 2),
        preferred_language VARCHAR(5) NOT NULL
    )""",
    """CREATE TABLE hw2_movies (
        movie_id BIGINT PRIMARY KEY,
        title TEXT NOT NULL,
        release_year INTEGER NOT NULL CHECK (release_year BETWEEN 1888 AND 2100),
        original_language VARCHAR(5) NOT NULL
    )""",
    """CREATE TABLE hw2_genres (
        genre_id INTEGER PRIMARY KEY,
        genre_name VARCHAR(50) NOT NULL UNIQUE,
        genre_group VARCHAR(30) NOT NULL,
        description TEXT NOT NULL
    )""",
    """CREATE TABLE hw2_ratings (
        rating_id BIGINT PRIMARY KEY,
        user_id BIGINT NOT NULL REFERENCES hw2_users(user_id),
        movie_id BIGINT NOT NULL REFERENCES hw2_movies(movie_id),
        rating NUMERIC(2,1) NOT NULL
            CHECK (rating BETWEEN 0.5 AND 5.0 AND rating * 2 = trunc(rating * 2)),
        rated_at TIMESTAMPTZ NOT NULL,
        UNIQUE (user_id, movie_id, rated_at)
    )""",
    """CREATE TABLE hw2_movie_genres (
        movie_id BIGINT NOT NULL REFERENCES hw2_movies(movie_id),
        genre_id INTEGER NOT NULL REFERENCES hw2_genres(genre_id),
        PRIMARY KEY (movie_id, genre_id)
    )""",
    """CREATE TABLE hw2_user_features_offline (
        user_id BIGINT NOT NULL REFERENCES hw2_users(user_id),
        ratings_count_30d INTEGER NOT NULL CHECK (ratings_count_30d >= 0),
        avg_rating_30d NUMERIC(3,2)
            CHECK (avg_rating_30d IS NULL OR avg_rating_30d BETWEEN 0.50 AND 5.00),
        favorite_genre_id INTEGER REFERENCES hw2_genres(genre_id),
        valid_from TIMESTAMPTZ NOT NULL,
        valid_to TIMESTAMPTZ,
        PRIMARY KEY (user_id, valid_from),
        CHECK (valid_to IS NULL OR valid_to > valid_from)
    )""",
    """CREATE TABLE hw2_training_events (
        event_id BIGINT PRIMARY KEY,
        user_id BIGINT NOT NULL REFERENCES hw2_users(user_id),
        movie_id BIGINT NOT NULL REFERENCES hw2_movies(movie_id),
        prediction_ts TIMESTAMPTZ NOT NULL,
        label_horizon_end TIMESTAMPTZ NOT NULL,
        target_observed_ts TIMESTAMPTZ NOT NULL,
        target_rating NUMERIC(2,1) NOT NULL CHECK (
            target_rating BETWEEN 0.5 AND 5.0
            AND target_rating * 2 = trunc(target_rating * 2)
        ),
        CHECK (label_horizon_end > prediction_ts),
        CHECK (target_observed_ts > prediction_ts),
        CHECK (target_observed_ts <= label_horizon_end)
    )""",
    """CREATE TABLE hw2_user_features_online (
        user_id BIGINT PRIMARY KEY REFERENCES hw2_users(user_id),
        ratings_count_30d INTEGER NOT NULL CHECK (ratings_count_30d >= 0),
        avg_rating_30d NUMERIC(3,2)
            CHECK (avg_rating_30d IS NULL OR avg_rating_30d BETWEEN 0.50 AND 5.00),
        favorite_genre_id INTEGER REFERENCES hw2_genres(genre_id),
        feature_ts TIMESTAMPTZ NOT NULL,
        updated_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
    )""",
    "CREATE INDEX hw2_idx_ratings_user_time ON hw2_ratings(user_id, rated_at DESC)",
    "CREATE INDEX hw2_idx_features_asof ON hw2_user_features_offline(user_id, valid_from DESC)",
    "CREATE INDEX hw2_idx_events_user_time ON hw2_training_events(user_id, prediction_ts)"
]
execute(*ddl)
show("""SELECT table_name FROM information_schema.tables
        WHERE table_schema = current_schema()
          AND table_name LIKE 'hw2_%'
        ORDER BY table_name""")

,table_name
0,hw2_genres
1,hw2_movie_genres
2,hw2_movies
3,hw2_ratings
4,hw2_training_events
5,hw2_user_features_offline
6,hw2_user_features_online
7,hw2_users


,table_name
0,hw2_genres
1,hw2_movie_genres
2,hw2_movies
3,hw2_ratings
4,hw2_training_events
5,hw2_user_features_offline
6,hw2_user_features_online
7,hw2_users


In [4]:
seed = [
    """INSERT INTO hw2_users (user_id, signup_date, country, preferred_language) VALUES
        (1, DATE '2023-01-10', 'UA', 'uk'),
        (2, DATE '2023-02-11', 'PL', 'pl'),
        (3, DATE '2023-03-12', 'UA', 'uk')""",
    """INSERT INTO hw2_movies (movie_id, title, release_year, original_language) VALUES
        (101, 'Sample Space Adventure', 2001, 'en'),
        (102, 'Sample Family Story', 2005, 'en'),
        (103, 'Sample Mystery', 2012, 'en'),
        (104, 'Sample Comedy', 2019, 'en')""",
    """INSERT INTO hw2_genres (genre_id, genre_name, genre_group, description) VALUES
        (1, 'Sci-Fi', 'Fiction', 'Speculative science and space stories'),
        (2, 'Drama', 'Narrative', 'Character-driven narratives'),
        (3, 'Comedy', 'Entertainment', 'Humorous narratives')""",
    """INSERT INTO hw2_movie_genres (movie_id, genre_id) VALUES
        (101,1), (101,2), (102,2), (103,1), (104,3)""",
    """INSERT INTO hw2_ratings (rating_id, user_id, movie_id, rating, rated_at) VALUES
        (1,1,101,4.0,TIMESTAMPTZ '2024-10-02 10:00:00+00'),
        (2,1,102,3.5,TIMESTAMPTZ '2024-10-16 10:00:00+00'),
        (3,1,103,4.5,TIMESTAMPTZ '2024-11-02 10:00:00+00'),
        (4,1,104,4.0,TIMESTAMPTZ '2024-11-17 10:00:00+00'),
        (5,2,101,3.0,TIMESTAMPTZ '2024-10-03 10:00:00+00'),
        (6,2,102,4.0,TIMESTAMPTZ '2024-10-17 10:00:00+00'),
        (7,2,103,4.5,TIMESTAMPTZ '2024-11-03 10:00:00+00'),
        (8,2,104,3.5,TIMESTAMPTZ '2024-11-18 10:00:00+00'),
        (9,3,101,5.0,TIMESTAMPTZ '2024-10-04 10:00:00+00'),
        (10,3,102,4.5,TIMESTAMPTZ '2024-10-18 10:00:00+00'),
        (11,3,103,4.0,TIMESTAMPTZ '2024-11-04 10:00:00+00'),
        (12,3,104,4.5,TIMESTAMPTZ '2024-11-19 10:00:00+00')""",
    """INSERT INTO hw2_user_features_offline
        (user_id, ratings_count_30d, avg_rating_30d, favorite_genre_id, valid_from, valid_to)
    VALUES
        (1, 1, 3.50, 2, TIMESTAMPTZ '2024-10-01 00:00:00+00', TIMESTAMPTZ '2024-10-15 00:00:00+00'),
        (1, 2, 3.75, 2, TIMESTAMPTZ '2024-10-15 00:00:00+00', TIMESTAMPTZ '2024-11-01 00:00:00+00'),
        (1, 3, 4.00, 1, TIMESTAMPTZ '2024-11-01 00:00:00+00', TIMESTAMPTZ '2024-11-15 00:00:00+00'),
        (1, 4, 4.25, 1, TIMESTAMPTZ '2024-11-15 00:00:00+00', NULL),
        (2, 1, 3.00, 2, TIMESTAMPTZ '2024-10-01 00:00:00+00', TIMESTAMPTZ '2024-10-15 00:00:00+00'),
        (2, 2, 3.25, 2, TIMESTAMPTZ '2024-10-15 00:00:00+00', TIMESTAMPTZ '2024-11-01 00:00:00+00'),
        (2, 3, 3.75, 1, TIMESTAMPTZ '2024-11-01 00:00:00+00', TIMESTAMPTZ '2024-11-15 00:00:00+00'),
        (2, 4, 4.00, 3, TIMESTAMPTZ '2024-11-15 00:00:00+00', NULL),
        (3, 1, 4.00, 1, TIMESTAMPTZ '2024-10-01 00:00:00+00', TIMESTAMPTZ '2024-10-15 00:00:00+00'),
        (3, 2, 4.25, 1, TIMESTAMPTZ '2024-10-15 00:00:00+00', TIMESTAMPTZ '2024-11-01 00:00:00+00'),
        (3, 3, 4.50, 2, TIMESTAMPTZ '2024-11-01 00:00:00+00', TIMESTAMPTZ '2024-11-15 00:00:00+00'),
        (3, 4, 4.75, 2, TIMESTAMPTZ '2024-11-15 00:00:00+00', NULL)""",
    """INSERT INTO hw2_training_events
        (event_id, user_id, movie_id, prediction_ts, label_horizon_end, target_observed_ts, target_rating)
    VALUES
        (1,1,103,TIMESTAMPTZ '2024-10-10 12:00:00+00',TIMESTAMPTZ '2024-11-10 12:00:00+00',TIMESTAMPTZ '2024-11-02 10:00:00+00',4.5),
        (2,1,104,TIMESTAMPTZ '2024-10-15 00:00:00+00',TIMESTAMPTZ '2024-11-25 00:00:00+00',TIMESTAMPTZ '2024-11-17 10:00:00+00',4.0),
        (3,1,104,TIMESTAMPTZ '2024-11-10 12:00:00+00',TIMESTAMPTZ '2024-11-25 12:00:00+00',TIMESTAMPTZ '2024-11-17 10:00:00+00',4.0),
        (4,2,104,TIMESTAMPTZ '2024-10-20 12:00:00+00',TIMESTAMPTZ '2024-11-25 12:00:00+00',TIMESTAMPTZ '2024-11-18 10:00:00+00',3.5),
        (5,2,104,TIMESTAMPTZ '2024-11-15 00:00:00+00',TIMESTAMPTZ '2024-11-30 00:00:00+00',TIMESTAMPTZ '2024-11-18 10:00:00+00',3.5),
        (6,3,102,TIMESTAMPTZ '2024-10-12 12:00:00+00',TIMESTAMPTZ '2024-10-27 12:00:00+00',TIMESTAMPTZ '2024-10-18 10:00:00+00',4.5),
        (7,3,104,TIMESTAMPTZ '2024-11-05 12:00:00+00',TIMESTAMPTZ '2024-11-25 12:00:00+00',TIMESTAMPTZ '2024-11-19 10:00:00+00',4.5)""",
    """INSERT INTO hw2_user_features_online
        (user_id, ratings_count_30d, avg_rating_30d, favorite_genre_id, feature_ts)
    SELECT user_id, ratings_count_30d, avg_rating_30d, favorite_genre_id, valid_from
    FROM hw2_user_features_offline
    WHERE valid_to IS NULL"""
]
execute(*seed)
show("""SELECT user_id, COUNT(*) AS versions
        FROM hw2_user_features_offline
        GROUP BY user_id ORDER BY user_id""")
show("SELECT * FROM hw2_user_features_online ORDER BY user_id")

,user_id,versions
0,1,4
1,2,4
2,3,4


,user_id,ratings_count_30d,avg_rating_30d,favorite_genre_id,feature_ts,updated_at
0,1,4,4.25,1,2024-11-15 00:00:00+00:00,2026-11-08 15:36:06.180243+00:00
1,2,4,4.00,3,2024-11-15 00:00:00+00:00,2026-11-08 15:36:06.180243+00:00
2,3,4,4.75,2,2024-11-15 00:00:00+00:00,2026-11-08 15:36:06.180243+00:00


,user_id,ratings_count_30d,avg_rating_30d,favorite_genre_id,feature_ts,updated_at
0,1,4,4.25,1,2024-11-15 00:00:00+00:00,2026-11-08 15:36:06.180243+00:00
1,2,4,4.00,3,2024-11-15 00:00:00+00:00,2026-11-08 15:36:06.180243+00:00
2,3,4,4.75,2,2024-11-15 00:00:00+00:00,2026-11-08 15:36:06.180243+00:00


In [5]:
asof_1 = show("""SELECT
    user_id, ratings_count_30d, avg_rating_30d,
    valid_from, valid_to
FROM hw2_user_features_offline
WHERE user_id = 1
  AND valid_from <= TIMESTAMPTZ '2024-10-10 12:00:00+00'
  AND (
      valid_to > TIMESTAMPTZ '2024-10-10 12:00:00+00'
      OR valid_to IS NULL
  )""")
assert len(asof_1) == 1 and int(asof_1.iloc[0]['ratings_count_30d']) == 1

,user_id,ratings_count_30d,avg_rating_30d,valid_from,valid_to
0,1,1,3.5,2024-10-01 00:00:00+00:00,2024-10-15 00:00:00+00:00


In [6]:
asof_2 = show("""SELECT
    user_id, ratings_count_30d, avg_rating_30d,
    valid_from, valid_to
FROM hw2_user_features_offline
WHERE user_id = 1
  AND valid_from <= TIMESTAMPTZ '2024-10-15 00:00:00+00'
  AND (
      valid_to > TIMESTAMPTZ '2024-10-15 00:00:00+00'
      OR valid_to IS NULL
  )""")
assert len(asof_2) == 1 and int(asof_2.iloc[0]['ratings_count_30d']) == 2

,user_id,ratings_count_30d,avg_rating_30d,valid_from,valid_to
0,1,2,3.75,2024-10-15 00:00:00+00:00,2024-11-01 00:00:00+00:00


In [7]:
training_sql = """SELECT
    e.event_id,
    e.user_id,
    e.movie_id,
    e.prediction_ts,
    e.label_horizon_end,
    e.target_observed_ts,
    e.target_rating,
    f.ratings_count_30d,
    f.avg_rating_30d,
    f.favorite_genre_id,
    f.valid_from AS feature_valid_from,
    f.valid_to AS feature_valid_to
FROM hw2_training_events AS e
LEFT JOIN hw2_user_features_offline AS f
    ON f.user_id = e.user_id
   AND f.valid_from <= e.prediction_ts
   AND (f.valid_to > e.prediction_ts OR f.valid_to IS NULL)
ORDER BY e.event_id"""
training = show(training_sql)
assert len(training) == 7
assert training['event_id'].is_unique
assert training['ratings_count_30d'].notna().all()
assert list(training['ratings_count_30d'].astype(int)) == [1,2,3,2,4,1,3]
print('PASS: 7 training events, one feature version per event; expected values verified.')

,event_id,user_id,movie_id,prediction_ts,label_horizon_end,target_observed_ts,target_rating,ratings_count_30d,avg_rating_30d,favorite_genre_id,feature_valid_from,feature_valid_to
0,1,1,103,2024-10-10 12:00:00+00:00,2024-11-10 12:00:00+00:00,2024-11-02 10:00:00+00:00,4.5,1,3.50,2,2024-10-01 00:00:00+00:00,2024-10-15 00:00:00+00:00
1,2,1,104,2024-10-15 00:00:00+00:00,2024-11-25 00:00:00+00:00,2024-11-17 10:00:00+00:00,4.0,2,3.75,2,2024-10-15 00:00:00+00:00,2024-11-01 00:00:00+00:00
2,3,1,104,2024-11-10 12:00:00+00:00,2024-11-25 12:00:00+00:00,2024-11-17 10:00:00+00:00,4.0,3,4.00,1,2024-11-01 00:00:00+00:00,2024-11-15 00:00:00+00:00
3,4,2,104,2024-10-20 12:00:00+00:00,2024-11-25 12:00:00+00:00,2024-11-18 10:00:00+00:00,3.5,2,3.25,2,2024-10-15 00:00:00+00:00,2024-11-01 00:00:00+00:00
4,5,2,104,2024-11-15 00:00:00+00:00,2024-11-30 00:00:00+00:00,2024-11-18 10:00:00+00:00,3.5,4,4.00,3,2024-11-15 00:00:00+00:00,NaT
5,6,3,102,2024-10-12 12:00:00+00:00,2024-10-27 12:00:00+00:00,2024-10-18 10:00:00+00:00,4.5,1,4.00,1,2024-10-01 00:00:00+00:00,2024-10-15 00:00:00+00:00
6,7,3,104,2024-11-05 12:00:00+00:00,2024-11-25 12:00:00+00:00,2024-11-19 10:00:00+00:00,4.5,3,4.50,2,2024-11-01 00:00:00+00:00,2024-11-15 00:00:00+00:00


PASS: 7 training events, one feature version per event; expected values verified.


In [8]:
show("""SELECT
    (SELECT COUNT(*) FROM hw2_training_events) AS expected_events,
    (SELECT COUNT(*) FROM hw2_training_events e
        JOIN hw2_user_features_offline f ON f.user_id = e.user_id) AS incorrect_join_rows,
    (SELECT COUNT(*) FROM hw2_training_events e
        LEFT JOIN hw2_user_features_offline f
          ON f.user_id = e.user_id
         AND f.valid_from <= e.prediction_ts
         AND (f.valid_to > e.prediction_ts OR f.valid_to IS NULL)
    ) AS correct_asof_rows""")

,expected_events,incorrect_join_rows,correct_asof_rows
0,7,28,7


,expected_events,incorrect_join_rows,correct_asof_rows
0,7,28,7


In [9]:
overlaps = show("""SELECT
    a.user_id,
    a.valid_from AS a_from, a.valid_to AS a_to,
    b.valid_from AS b_from, b.valid_to AS b_to
FROM hw2_user_features_offline a
JOIN hw2_user_features_offline b
  ON a.user_id = b.user_id
 AND a.valid_from < COALESCE(b.valid_to, 'infinity'::timestamptz)
 AND b.valid_from < COALESCE(a.valid_to, 'infinity'::timestamptz)
 AND a.valid_from < b.valid_from
ORDER BY a.user_id, a.valid_from""")
assert overlaps.empty, 'Overlapping feature intervals found'
print('PASS: overlapping intervals = 0')

,user_id,a_from,a_to,b_from,b_to


PASS: overlapping intervals = 0


In [10]:
execute("""INSERT INTO hw2_user_features_online AS dst
    (user_id, ratings_count_30d, avg_rating_30d, favorite_genre_id, feature_ts)
SELECT user_id, ratings_count_30d, avg_rating_30d, favorite_genre_id, valid_from
FROM hw2_user_features_offline
WHERE valid_to IS NULL
ON CONFLICT (user_id) DO UPDATE SET
    ratings_count_30d = EXCLUDED.ratings_count_30d,
    avg_rating_30d = EXCLUDED.avg_rating_30d,
    favorite_genre_id = EXCLUDED.favorite_genre_id,
    feature_ts = EXCLUDED.feature_ts,
    updated_at = CURRENT_TIMESTAMP
WHERE dst.feature_ts <= EXCLUDED.feature_ts""")
show("""SELECT user_id, ratings_count_30d, avg_rating_30d, feature_ts
        FROM hw2_user_features_online ORDER BY user_id""")

,user_id,ratings_count_30d,avg_rating_30d,feature_ts
0,1,4,4.25,2024-11-15 00:00:00+00:00
1,2,4,4.00,2024-11-15 00:00:00+00:00
2,3,4,4.75,2024-11-15 00:00:00+00:00


,user_id,ratings_count_30d,avg_rating_30d,feature_ts
0,1,4,4.25,2024-11-15 00:00:00+00:00
1,2,4,4.00,2024-11-15 00:00:00+00:00
2,3,4,4.75,2024-11-15 00:00:00+00:00


In [11]:
checks = show("""SELECT
    (SELECT COUNT(*) FROM information_schema.tables
     WHERE table_schema = current_schema() AND table_name LIKE 'hw2_%') AS table_count,
    (SELECT COUNT(*) FROM hw2_user_features_offline) AS offline_versions,
    (SELECT COUNT(DISTINCT user_id) FROM hw2_user_features_offline) AS feature_users,
    (SELECT MAX(cnt) FROM (
        SELECT COUNT(*) cnt FROM hw2_user_features_offline GROUP BY user_id
    ) x) AS max_versions_per_user,
    (SELECT COUNT(*) FROM hw2_training_events) AS training_events,
    (SELECT COUNT(*) FROM hw2_user_features_online) AS online_rows""")
c = checks.iloc[0]
assert int(c.table_count) == 8
assert int(c.offline_versions) >= 10
assert int(c.feature_users) >= 3
assert int(c.max_versions_per_user) >= 3
assert int(c.training_events) == len(training)
assert int(c.online_rows) == 3
assert overlaps.empty and training.event_id.is_unique
label_checks = show("""SELECT e.event_id, r.rating_id
FROM hw2_training_events e
LEFT JOIN hw2_ratings r
  ON r.user_id = e.user_id AND r.movie_id = e.movie_id
 AND r.rated_at = e.target_observed_ts AND r.rating = e.target_rating
WHERE r.rating_id IS NULL""")
assert label_checks.empty, 'Training labels must match observed ratings'
print('ALL ACCEPTANCE CHECKS PASSED')

,table_count,offline_versions,feature_users,max_versions_per_user,training_events,online_rows
0,8,12,3,4,7,3


,event_id,rating_id


ALL ACCEPTANCE CHECKS PASSED
